# MAI 600 — Module 3: Attention Walkthrough
## Financial Fraud Alert Example

**Domain:** Financial services / fraud operations (AI in finance and business)

This notebook walks through how a Transformer processes a short fraud-investigation summary: how the raw text becomes tokens, how token order is preserved, how self-attention connects tokens to one another, and how those connections produce an output.

**Scope note:** the goal here is not to train a model. It is to show that I understand how tokenization, embeddings, positional encoding, Q/K/V, attention weights, multi-head attention, feed-forward layers, and output probabilities fit together — and to document specific attention behaviors in my chosen text.

## 1. Import libraries

In [ ]:
!pip install -q tiktoken

In [ ]:
import numpy as np
import pandas as pd
import tiktoken

np.random.seed(0)
print("libraries loaded")

## 2. Load the text sample

**Part 1 requirement — select and explain the text.**

I wrote this paragraph myself as a fictional fraud-investigation summary. It contains no real customer data, no account numbers, and no identifying information, so it is safe to publish on GitHub.

In [ ]:
TEXT = (
"The compliance officer flagged the international wire transfer because it originated "
"from a business account that had been dormant for nearly eight months. The account holder "
"stated that they had not authorized the payment and did not recognize the receiving institution. "
"The fraud response team froze the account temporarily, reversed the pending transfer, and "
"required identity re-verification before restoring online access. After reviewing the full "
"transaction history, they confirmed that the activity was inconsistent with the customer\'s "
"established spending pattern. The investigation also revealed that the login preceding the "
"request came from an unfamiliar device and region, although no additional unauthorized "
"transfers were completed. Because the transfer was caught before settlement, the bank "
"recovered the funds and closed the case without a loss."
)

print(TEXT)
print()
print("word count:", len(TEXT.split()))
print("sentence count:", TEXT.count("."))

### Why this text works for studying attention

| Item | Response |
|---|---|
| **Domain** | Financial services — fraud operations |
| **Text type** | Fraud investigation / case summary |
| **Why this text works** | It is dense with references that cannot be resolved from the words alone. It has two pronouns pointing at different entities, a cause/effect chain, a contrast clause, and a long-range reference back to the first sentence. |
| **Key relationships the model must track** | "it" → the wire transfer (not the officer); the first "they" → the account holder; the second "they" → the fraud response team; "the activity" → the transfer described three sentences earlier; "because"/"although" linking cause and contrast. |

The most interesting property: **the token "they" appears twice and refers to a different entity each time.** Nothing about the word itself distinguishes them — the model has to use position and surrounding context. That is exactly the job self-attention does.

## 3. Tokenize the text

**Part 2 requirement — tokenization and context setup.**

I'm using `cl100k_base`, the tokenizer behind GPT-3.5/GPT-4, so the token counts here are real rather than approximated.

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")

ids = enc.encode(TEXT)
tokens = [enc.decode([i]) for i in ids]

print("words :", len(TEXT.split()))
print("tokens:", len(ids))
print("token-to-word ratio:", round(len(ids)/len(TEXT.split()), 2))
print()
print("First 30 tokens:")
print(tokens[:30])

### How do individual terms tokenize?

In [ ]:
probe = ["dormant", "settlement", "unauthorized", "re-verification",
         "compliance", "customer's", "wire transfer", " it", " they"]

rows = []
for w in probe:
    t = enc.encode(w)
    rows.append({
        "term": w,
        "n_tokens": len(t),
        "pieces": " | ".join(enc.decode([i]) for i in t)
    })

pd.DataFrame(rows)

**What stands out:**

- `dormant` — a normal English word — splits into **three** pieces (`d` / `orm` / `ant`). It just isn't frequent enough in the training text to earn its own token.
- `settlement` and `unauthorized` each split into two pieces, even though both are common in finance writing.
- `re-verification` becomes three tokens because the hyphen is split out on its own.
- Meanwhile `it` and `they` — the two words that carry the hardest interpretive work in this paragraph — are **single tokens each**.

That last point is the one worth sitting with: token length has nothing to do with semantic difficulty. The cheapest tokens in the paragraph are the ones the model has to work hardest to resolve.

### Locate the reference words by position

In [ ]:
targets = [" it", " they", " activity", " although", " because"]

for t in targets:
    pos = [i for i, tok in enumerate(tokens) if tok == t]
    print(f"{t!r:12} appears at token position(s): {pos}")

### Required token table

| Token / Phrase | Why It Matters |
|---|---|
| `wire transfer` (pos 6–7) | The central event of the paragraph; the anchor that later references point back to |
| `it` (pos 9) | Must resolve to the *transfer*, not to the *compliance officer* — both are plausible grammatically |
| `they` (pos 29) | Refers to the **account holder** |
| `they` (pos 75) | Same token, but refers to the **fraud response team** — resolved only by position and context |
| `dormant` | Risk indicator; also tokenizes into 3 subword pieces |
| `the activity` (pos 79) | A long-range reference back to the transfer in sentence 1 |
| `because` (pos 8) | Signals a cause/effect link |
| `although` (pos 108) | Signals a contrast — the finding is suspicious *but* limited |

### Why order matters

Positional information is not optional here. Strip the ordering out and the two occurrences of `they` become literally identical inputs — the same token ID, the same embedding vector. The only thing that separates "the account holder" from "the fraud response team" is *where in the sequence each one sits*. That is why positional encoding gets added to the embeddings before attention runs: without it, the model would have no way to tell the two apart.

## 4. How self-attention actually works

Below is scaled dot-product attention implemented from scratch, so the mechanism is visible rather than hidden behind a library call.

**Honesty note about what this does and does not show:** the projection matrices below are **random**, not trained. So the attention pattern this produces is *not* linguistically meaningful — it demonstrates the machinery (shapes, the softmax, the weighted sum), not learned behavior. Real pronoun resolution comes from weights learned over billions of tokens. The attention behaviors in Part 5 are therefore documented by linguistic annotation, not read off this toy model. I think that distinction matters and is worth stating plainly.

In [ ]:
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V):
    """
    Q, K, V shape: (n_heads, seq_len, d_head)
    Returns the attended output and the attention weight matrix.
    """
    d_k = Q.shape[-1]
    scores  = Q @ K.transpose(0, 2, 1) / np.sqrt(d_k)   # (heads, seq, seq)
    weights = softmax(scores, axis=-1)                   # each row sums to 1
    return weights @ V, weights


def multi_head_attention(X, n_heads=4, d_model=32):
    """X shape: (seq_len, d_model)"""
    seq_len = X.shape[0]
    d_head  = d_model // n_heads

    # Learned in a real model; random here.
    W_Q = np.random.randn(d_model, d_model) * 0.1
    W_K = np.random.randn(d_model, d_model) * 0.1
    W_V = np.random.randn(d_model, d_model) * 0.1
    W_O = np.random.randn(d_model, d_model) * 0.1

    # Project, then split into heads
    Q = (X @ W_Q).reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)
    K = (X @ W_K).reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)
    V = (X @ W_V).reshape(seq_len, n_heads, d_head).transpose(1, 0, 2)

    out, weights = scaled_dot_product_attention(Q, K, V)

    # Concatenate heads back together, then final projection
    out = out.transpose(1, 0, 2).reshape(seq_len, d_model) @ W_O
    return out, weights


print("attention functions defined")

### Run it on a slice of the sequence

In [ ]:
SEQ, D_MODEL, N_HEADS = 12, 32, 4

# Stand-in embeddings (a real model looks these up from a trained embedding table)
X = np.random.randn(SEQ, D_MODEL) * 0.5

output, attn_weights = multi_head_attention(X, N_HEADS, D_MODEL)

print("input embeddings X :", X.shape,            "(seq_len, d_model)")
print("attention weights  :", attn_weights.shape, "(n_heads, seq_len, seq_len)")
print("output             :", output.shape,       "(seq_len, d_model)")
print()
print("Every attention row sums to 1.0:", np.allclose(attn_weights.sum(axis=-1), 1.0))
print()
print("Head 0, how token 0 distributes its attention across all 12 tokens:")
print(np.round(attn_weights[0, 0], 3))

Notice that those weights come out close to uniform (roughly `1/12 ≈ 0.083` each). That is the expected result and it is informative: **an untrained attention head spreads its attention evenly and learns nothing.** The sharp, selective patterns you see in published attention visualizations — a pronoun locking onto its antecedent — are entirely a product of training. The architecture supplies the capacity; the weights supply the behavior.

### Adding positional information

In [ ]:
def positional_encoding(seq_len, d_model):
    """Sinusoidal positional encoding from Vaswani et al. (2017)."""
    pos = np.arange(seq_len)[:, None]
    i   = np.arange(d_model)[None, :]
    angle = pos / np.power(10000, (2 * (i // 2)) / d_model)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe


pe = positional_encoding(SEQ, D_MODEL)
X_with_pos = X + pe          # this sum is what actually enters the first block

print("positional encoding:", pe.shape)
print()
# Two identical embeddings at different positions become distinguishable
same_token = np.ones(D_MODEL) * 0.4
print("Identical token vector placed at position 3 vs position 9:")
print("  cosine similarity BEFORE adding position:", 1.0)
a, b = same_token + pe[3], same_token + pe[9]
cos = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print("  cosine similarity AFTER  adding position:", round(float(cos), 4))

This is the mechanism that makes the two `they` tokens separable. Before positional encoding they are the same vector — cosine similarity 1.0, completely indistinguishable. After the positional vector is added, they are measurably different, and attention can treat them as different things.

## 5. Attention behaviors in this text

**Part 3 requirement — document at least three attention behaviors.** I documented five.

These are linguistic annotations of what the model *must* do to read this paragraph correctly — not readings off the untrained toy model above.

In [ ]:
behaviors = [
    {
        "Attention Behavior": "Pronoun resolution",
        "Token / Phrase 1": "it (pos 9)",
        "Token / Phrase 2": "wire transfer (pos 6-7)",
        "Why the Relationship Matters":
            "Grammatically 'it' could attach to 'the compliance officer', but only the "
            "transfer can 'originate from an account'. The model must use meaning, not "
            "just proximity, to pick the right antecedent."
    },
    {
        "Attention Behavior": "Entity tracking (same token, two referents)",
        "Token / Phrase 1": "they (pos 29) / they (pos 75)",
        "Token / Phrase 2": "account holder / fraud response team",
        "Why the Relationship Matters":
            "The identical token resolves to two different entities. Position and "
            "surrounding context are the only signals available. Getting this wrong "
            "would flip who reported the problem and who investigated it."
    },
    {
        "Attention Behavior": "Cause and effect",
        "Token / Phrase 1": "because ... dormant (pos 8-18)",
        "Token / Phrase 2": "flagged (pos 3)",
        "Why the Relationship Matters":
            "The reason for the flag arrives after the action. The model has to hold "
            "'flagged' open and bind the justification to it."
    },
    {
        "Attention Behavior": "Long-range dependency",
        "Token / Phrase 1": "the activity (pos 79)",
        "Token / Phrase 2": "wire transfer (pos 6-7)",
        "Why the Relationship Matters":
            "A reference roughly 70 tokens back, across three sentence boundaries. "
            "This is precisely where older recurrent models degraded and where "
            "self-attention's direct token-to-token connections help."
    },
    {
        "Attention Behavior": "Contrast",
        "Token / Phrase 1": "although (pos 108)",
        "Token / Phrase 2": "no additional unauthorized transfers",
        "Why the Relationship Matters":
            "'although' reverses the expected conclusion. Miss it and the summary "
            "becomes 'a breach occurred' instead of 'suspicious activity, contained "
            "with no further loss' - a materially different finding."
    },
]

pd.set_option("display.max_colwidth", 90)
pd.DataFrame(behaviors)

## 6. Save results

In [ ]:
import os
os.makedirs("results", exist_ok=True)

pd.DataFrame(behaviors).to_csv("results/attention_behaviors.csv", index=False)

token_table = pd.DataFrame({
    "position": range(len(tokens)),
    "token": tokens,
    "token_id": ids,
})
token_table.to_csv("results/tokens.csv", index=False)

print("saved results/attention_behaviors.csv")
print("saved results/tokens.csv")
token_table.head(12)

## 7. Reflection

The thing that reframed this for me was realizing that **the hardest words in my paragraph are the cheapest tokens.** `it` and `they` are one token each — as cheap as it gets — while `dormant` costs three. But `dormant` means the same thing everywhere it appears, and `they` means two entirely different things 46 tokens apart in the same paragraph. Token cost and interpretive difficulty are unrelated, and I don't think I would have noticed that without laying the tokens out by position.

Building attention from scratch also made the Q/K/V split concrete in a way the formula alone never did. Query as "what am I looking for," key as "what do I match against," value as "what I actually pass along" — three different jobs, which is why you need three separate projections of the same vector instead of one. And the near-uniform output from my untrained version was the useful part of the exercise: the architecture on its own does nothing. The published attention heatmaps where a pronoun snaps to its antecedent are showing learned weights, not the mechanism.

The positional encoding demonstration was the other piece that landed. Two identical tokens are the identical vector — cosine similarity exactly 1.0 — until position is added. Everything the model does with my two `they` tokens depends on that one addition happening before attention runs.

One caution I'd carry into anything applied: attention weights are often described as an explanation of model behavior, and that framing seems too generous. A weight says one token influenced another; it doesn't say why, and multiple heads across many layers combine in ways a single heatmap can't capture. For a fraud workflow like the one in my text, I'd treat attention patterns as a debugging aid, not as an audit trail I'd defend to a compliance reviewer.